# 08 Estimate Noisy Semi-Synthetic Utility Specification

This notebook estimates the fine-grained semi-synthetic true utility surface for the 1,777 structural-estimation users.

Notebook 07 produced observed posterior expected utility targets:

$$
q^{\mathrm{target}}_{itj}=\frac{EU_{itj}-\hat\mu_{i,k_1(j)}}{\hat\sigma_{i,k_1(j)}}.
$$

The parameters of interest in this notebook are personalized level-2 and level-3 fixed effects:

$$
\alpha_{i,k_2}\quad\text{and}\quad\delta_{i,k_3}.
$$

For each user-video pair, indexed by:

$$
(i,j),
$$

the systematic raw utility signal is:

$$
s_{ij}=\alpha_{i,k_2(j)}+\delta_{i,k_3(j)}.
$$

The raw signal is normalized over the full video catalog within each user × level-1 category:

$$
q^{\mathrm{signal}}_{ij}=\frac{s_{ij}-m_{i,k_1(j)}}{d_{i,k_1(j)}}.
$$

The denominator below is only the raw-score normalization denominator:

$$
d_{ik}.
$$

It is not the calibrated utility scale. The calibrated utility scale remains the structural-model estimate:

$$
\hat\sigma_{ik}.
$$

To make the semi-synthetic environment less mechanically deterministic, the final true utility also includes a small fixed random user-video utility shock. Let:

$$
\eta=\texttt{RANDOM\_UTILITY\_VARIANCE\_SHARE}.
$$

The default is:

$$
\eta=0.05.
$$

This means 5% of the within-user × level-1 standardized utility variance is assigned to random user-video noise before final re-standardization. Setting the share to zero recovers the old deterministic utility surface:

$$
\eta=0.
$$

For one fixed draw:

$$
\epsilon_{ij}\sim N(0,1),
$$

standardize the shock within each user × level-1 category to obtain:

$$
\epsilon^{\mathrm{std}}_{ij}.
$$

Then combine signal and noise:

$$
q^{\mathrm{raw}}_{ij}=\sqrt{1-\eta}\,q^{\mathrm{signal}}_{ij}+\sqrt{\eta}\,\epsilon^{\mathrm{std}}_{ij}.
$$

Because the signal and random component may be correlated in a finite catalog, re-standardize the raw combined score:

$$
q^{\mathrm{raw}}_{ij}
$$

within each user × level-1 category:

$$
q^{\mathrm{true}}_{ij}=\operatorname{standardize}_{j\in V_{k_1(j)}}\left(q^{\mathrm{raw}}_{ij}\right).
$$

The final semi-synthetic true utility is:

$$
u^*_{ij}=\hat\mu_{i,k_1(j)}+\hat\sigma_{i,k_1(j)}q^{\mathrm{true}}_{ij}.
$$

This final standardization guarantees that for every user and level-1 category:

$$
E_{j\in V_k}[u^*_{ij}]=\hat\mu_{ik},
\qquad
\operatorname{Var}_{j\in V_k}(u^*_{ij})=\hat\sigma_{ik}^2.
$$

## 1. Setup

In [1]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
import torch
from torch import nn

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
OUTPUT_DIR = PROJECT_ROOT / "python_code_new" / "outputs"
PROCESSED_DATA_PATH = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed" / "processed_data.parquet"

STRUCTURAL_PARAM_PATH = OUTPUT_DIR / "structural_estimates_theta_phi.npz"
STRUCTURAL_ARRAYS_PATH = OUTPUT_DIR / "structural_estimation_arrays.npz"
OBSERVED_TARGET_PATH = OUTPUT_DIR / "semi_synth_observed_targets.parquet"

PARAM_OUTPUT_NPZ = OUTPUT_DIR / "semi_synth_utility_params.npz"
CHECKPOINT_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_utility_fe_model.pt"
OBSERVED_UTILITY_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_utility_observed.parquet"
TRUE_UTILITY_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_true_utility.parquet"
DIAGNOSTIC_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_utility_diagnostics.json"
LOSS_HISTORY_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_utility_loss_history.csv"
BINNED_FIT_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_utility_binned_fit.csv"
CATALOG_OUTPUT_MODE = "single_file"  # choices: "single_file", "partitioned_by_user"

required_paths = [STRUCTURAL_PARAM_PATH, STRUCTURAL_ARRAYS_PATH, OBSERVED_TARGET_PATH, PROCESSED_DATA_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

SEED = 20260622
DEVICE = "cpu"
DTYPE = torch.float32

NUM_EPOCHS = 300
LEARNING_RATE = 1e-2
WEIGHT_DECAY = 0.0
LAMBDA_ALPHA = 0.0  # AdamW weight_decay is the default ridge penalty.
LAMBDA_DELTA = 0.0
INIT_SCALE = 0.01
SD_EPS = 1e-6
RANDOM_UTILITY_VARIANCE_SHARE = 0.05
RANDOM_UTILITY_SEED = SEED + 808
VALIDATION_SHARE = 0.05
USE_CONFIDENCE_WEIGHTS = False
LOG_EVERY = 10
EARLY_STOPPING_PATIENCE = 40
MIN_DELTA = 1e-6
WRITE_FULL_CATALOG_OUTPUT = True
CATALOG_USER_CHUNK_SIZE = 64

if not (0.0 <= RANDOM_UTILITY_VARIANCE_SHARE < 1.0):
    raise ValueError("RANDOM_UTILITY_VARIANCE_SHARE must be in [0, 1)")

np.random.seed(SEED)
torch.manual_seed(SEED)
print("device:", DEVICE)

device: cpu


## 2. Load Structural Parameters and Reconstruct

This section reconstructs the structural utility mean matrix:

$$
\hat\mu.
$$

In [2]:
params = np.load(STRUCTURAL_PARAM_PATH)
arrays = np.load(STRUCTURAL_ARRAYS_PATH)

user_ids = params["user_ids"].astype(np.int64)
cat_ids = params["cat_ids"].astype(np.int64)
reference_cat_idx = int(params["reference_cat_idx"][0])

bundle_user_ids = arrays["user_ids"].astype(np.int64)
bundle_cat_ids = arrays["cat_ids"].astype(np.int64)
if not np.array_equal(user_ids, bundle_user_ids):
    raise ValueError("user_ids mismatch between structural params and arrays")
if not np.array_equal(cat_ids, bundle_cat_ids):
    raise ValueError("cat_ids mismatch between structural params and arrays")

sigmas_by_user = arrays["sigmas_by_user"].astype(np.float32)
N_users = len(user_ids)
K_cat = len(cat_ids)

def theta_to_mu_matrix(params, center_reference=True):
    mu = params["theta_b_cat"].astype(np.float64)[None, :] + params["theta_U"].astype(np.float64) @ params["theta_V"].astype(np.float64).T
    if center_reference:
        mu = mu - mu[:, [reference_cat_idx]]
    return mu.astype(np.float32)

mu_matrix = theta_to_mu_matrix(params)
if mu_matrix.shape != sigmas_by_user.shape:
    raise ValueError(f"mu/sigma shape mismatch: {mu_matrix.shape} vs {sigmas_by_user.shape}")
if not np.isfinite(mu_matrix).all() or not np.isfinite(sigmas_by_user).all() or not (sigmas_by_user > 0).all():
    raise ValueError("Nonfinite or nonpositive structural mu/sigma values")

user_id_to_idx = {int(v): idx for idx, v in enumerate(user_ids)}
cat_id_to_idx = {int(v): idx for idx, v in enumerate(cat_ids)}

print("N_users:", N_users)
print("K_cat:", K_cat)
print("reference category:", int(cat_ids[reference_cat_idx]), "at idx", reference_cat_idx)
print("mu range:", np.percentile(mu_matrix, [0, 1, 50, 99, 100]))
print("sigma range:", np.percentile(sigmas_by_user, [0, 1, 50, 99, 100]))

N_users: 1777
K_cat: 39
reference category: -124 at idx 0
mu range: [-11.54083252  -1.63535028   1.31779826   4.59094415   9.52878857]
sigma range: [0.05373278 0.40681723 1.         1.58392408 3.81051183]


## 3. Build Full Video Catalog

The normalization must use all catalog videos within each level-1 category, not just observed interactions. If a video has conflicting category rows in the processed data, this cell uses the modal category and records the conflict count for diagnostics.

In [3]:
UNKNOWN_LEVEL2_ID = -1_000_002
UNKNOWN_LEVEL3_ID = -1_000_003

processed_cols = ["video_id", "i_cat_level1_id", "i_cat_level2_id", "i_cat_level3_id"]
processed_cat = pd.read_parquet(PROCESSED_DATA_PATH, columns=processed_cols)

for col, unknown in [("i_cat_level2_id", UNKNOWN_LEVEL2_ID), ("i_cat_level3_id", UNKNOWN_LEVEL3_ID)]:
    processed_cat[col] = processed_cat[col].fillna(unknown)

for col in processed_cols:
    processed_cat[col] = processed_cat[col].astype(np.int64)

conflict_counts = {}
for col in ["i_cat_level1_id", "i_cat_level2_id", "i_cat_level3_id"]:
    n_unique = processed_cat.groupby("video_id", observed=True)[col].nunique(dropna=False)
    conflict_counts[col] = int((n_unique > 1).sum())

def modal_int(x):
    mode = x.mode(dropna=False)
    return int(mode.iloc[0])

catalog_df = (
    processed_cat
    .groupby("video_id", as_index=False, observed=True)
    .agg(
        i_cat_level1_id=("i_cat_level1_id", modal_int),
        i_cat_level2_id=("i_cat_level2_id", modal_int),
        i_cat_level3_id=("i_cat_level3_id", modal_int),
    )
)

catalog_df = catalog_df.loc[catalog_df["i_cat_level1_id"].isin(set(cat_ids.tolist()))].copy()
catalog_df["cat_idx_int"] = catalog_df["i_cat_level1_id"].map(cat_id_to_idx).astype(np.int64)
catalog_df = catalog_df.sort_values(["cat_idx_int", "video_id"], kind="mergesort").reset_index(drop=True)
catalog_df["video_idx"] = np.arange(len(catalog_df), dtype=np.int64)

level2_ids = np.array(sorted(catalog_df["i_cat_level2_id"].unique().astype(np.int64)))
level3_ids = np.array(sorted(catalog_df["i_cat_level3_id"].unique().astype(np.int64)))
level2_id_to_idx = {int(v): idx for idx, v in enumerate(level2_ids)}
level3_id_to_idx = {int(v): idx for idx, v in enumerate(level3_ids)}

catalog_df["level2_idx"] = catalog_df["i_cat_level2_id"].map(level2_id_to_idx).astype(np.int64)
catalog_df["level3_idx"] = catalog_df["i_cat_level3_id"].map(level3_id_to_idx).astype(np.int64)

video_id_to_idx = dict(zip(catalog_df["video_id"].astype(int), catalog_df["video_idx"].astype(int)))

videos_per_level1 = catalog_df.groupby("i_cat_level1_id", observed=True).size().rename("n_videos").reset_index()
near_tiny_groups = videos_per_level1.loc[videos_per_level1["n_videos"].lt(2)]
if len(near_tiny_groups):
    raise ValueError("At least one level-1 category has fewer than 2 catalog videos; unit-variance normalization is not possible")

print("catalog videos:", len(catalog_df))
print("level-2 categories:", len(level2_ids))
print("level-3 categories:", len(level3_ids))
print("category conflict counts:", conflict_counts)
display(videos_per_level1.sort_values("n_videos").head(10))

catalog videos: 10728
level-2 categories: 139
level-3 categories: 221
category conflict counts: {'i_cat_level1_id': 0, 'i_cat_level2_id': 0, 'i_cat_level3_id': 0}


,i_cat_level1_id,n_videos
36,37,19
22,22,31
24,24,41
27,27,42
0,-124,43
35,36,49
23,23,50
21,21,50
14,14,60
38,39,62


## 4. Build Observed Training Rows

The training target from notebook 07 is:

$$
q^{\mathrm{target}}_{itj}.
$$

The fixed-effect model is fitted by matching the systematic signal:

$$
q^{\mathrm{signal}}_{ij}
$$

to the observed target:

$$
q^{\mathrm{target}}_{itj}
$$

on observed interactions only.

The model never uses `EU`, `r_hat`, watch ratio, completion, or the later random shock as input features. Those columns are targets or diagnostics only.

In [4]:
target_cols = [
    "row_id", "user_id", "video_id", "i_cat_level1_id", "user_idx_int", "cat_idx_int",
    "r_hat", "tau_hat", "mu_hat", "sigma_hat", "EU", "q_target", "posterior_confidence_weight",
]
observed_df = pd.read_parquet(OBSERVED_TARGET_PATH, columns=target_cols)

observed_df = observed_df.merge(
    catalog_df[["video_id", "video_idx", "i_cat_level2_id", "i_cat_level3_id", "level2_idx", "level3_idx", "cat_idx_int"]].rename(columns={"cat_idx_int": "catalog_cat_idx_int"}),
    on="video_id",
    how="left",
    validate="many_to_one",
)

valid_mask = (
    observed_df["video_idx"].notna()
    & observed_df["user_idx_int"].notna()
    & observed_df["cat_idx_int"].notna()
    & observed_df["q_target"].notna()
    & np.isfinite(observed_df["q_target"].to_numpy(dtype=np.float64))
    & np.isfinite(observed_df["posterior_confidence_weight"].to_numpy(dtype=np.float64))
    & observed_df["sigma_hat"].gt(0)
)
level1_match = observed_df["cat_idx_int"].eq(observed_df["catalog_cat_idx_int"])
valid_mask = valid_mask & level1_match
n_dropped = int((~valid_mask).sum())
if n_dropped:
    print("dropped observed rows:", n_dropped)
observed_df = observed_df.loc[valid_mask].copy().reset_index(drop=True)

obs_user_idx_np = observed_df["user_idx_int"].to_numpy(dtype=np.int64)
obs_video_idx_np = observed_df["video_idx"].to_numpy(dtype=np.int64)
q_target_np = observed_df["q_target"].to_numpy(dtype=np.float32)
obs_weight_np = observed_df["posterior_confidence_weight"].to_numpy(dtype=np.float32) if USE_CONFIDENCE_WEIGHTS else np.ones(len(observed_df), dtype=np.float32)
obs_weight_np = np.maximum(obs_weight_np, 1e-6).astype(np.float32)

rng = np.random.default_rng(SEED)
val_mask_np = rng.random(len(observed_df)) < VALIDATION_SHARE
if val_mask_np.sum() == 0 or (~val_mask_np).sum() == 0:
    raise ValueError("Validation split is empty; adjust VALIDATION_SHARE")
train_idx_np = np.flatnonzero(~val_mask_np).astype(np.int64)
val_idx_np = np.flatnonzero(val_mask_np).astype(np.int64)

print("observed rows for training:", len(observed_df))
print("train rows:", len(train_idx_np))
print("validation rows:", len(val_idx_np))
print("dropped rows:", n_dropped)
print("q_target summary:")
display(observed_df["q_target"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

observed rows for training: 4325560
train rows: 4109395
validation rows: 216165
dropped rows: 0
q_target summary:


count    4.325560e+06
mean     9.369274e-02
std      4.646544e-01
min     -3.389709e+00
1%      -1.246360e+00
5%      -8.831822e-01
50%      2.348497e-01
95%      6.304297e-01
99%      7.495593e-01
max      9.106165e-01
Name: q_target, dtype: float64

## 5. Define Fixed-Effect Model

For each user-video pair, the systematic raw score is:

$$
s_{ij}=\alpha_{i,k_2(j)}+\delta_{i,k_3(j)}.
$$

The estimated parameters of interest are the personalized level-2 and level-3 fixed effects:

$$
\alpha_{i,k_2},\qquad \delta_{i,k_3}.
$$

For every user and level-1 category, compute over the full video universe:

$$
V_k.
$$

The raw-score mean and standard deviation are:

$$
m_{ik}=\frac{1}{|V_k|}\sum_{v\in V_k}s_{iv},
\qquad
 d_{ik}=\sqrt{\frac{1}{|V_k|}\sum_{v\in V_k}(s_{iv}-m_{ik})^2}.
$$

Then the systematic utility score is:

$$
q^{\mathrm{signal}}_{ij}=\frac{s_{ij}-m_{i,k_1(j)}}{\max(d_{i,k_1(j)},\epsilon)}.
$$

The normalization is over the full catalog within each user × level-1 category, not over observed rows.

In [5]:
catalog_level1_idx_np = catalog_df["cat_idx_int"].to_numpy(dtype=np.int64)
catalog_level2_idx_np = catalog_df["level2_idx"].to_numpy(dtype=np.int64)
catalog_level3_idx_np = catalog_df["level3_idx"].to_numpy(dtype=np.int64)

catalog_level1_idx = torch.as_tensor(catalog_level1_idx_np, dtype=torch.long, device=DEVICE)
catalog_level2_idx = torch.as_tensor(catalog_level2_idx_np, dtype=torch.long, device=DEVICE)
catalog_level3_idx = torch.as_tensor(catalog_level3_idx_np, dtype=torch.long, device=DEVICE)

obs_user_idx = torch.as_tensor(obs_user_idx_np, dtype=torch.long, device=DEVICE)
obs_video_idx = torch.as_tensor(obs_video_idx_np, dtype=torch.long, device=DEVICE)
q_target = torch.as_tensor(q_target_np, dtype=DTYPE, device=DEVICE)
obs_weight = torch.as_tensor(obs_weight_np, dtype=DTYPE, device=DEVICE)
train_idx = torch.as_tensor(train_idx_np, dtype=torch.long, device=DEVICE)
val_idx = torch.as_tensor(val_idx_np, dtype=torch.long, device=DEVICE)

level1_video_indices = [
    torch.as_tensor(np.flatnonzero(catalog_level1_idx_np == k), dtype=torch.long, device=DEVICE)
    for k in range(K_cat)
]

class UtilityFixedEffectModel(nn.Module):
    def __init__(self, n_users, n_level2, n_level3, init_scale=0.01):
        super().__init__()
        self.alpha = nn.Parameter(init_scale * torch.randn(n_users, n_level2, dtype=DTYPE))
        self.delta = nn.Parameter(init_scale * torch.randn(n_users, n_level3, dtype=DTYPE))

    def raw_scores(self):
        return self.alpha[:, catalog_level2_idx] + self.delta[:, catalog_level3_idx]

    def normalized_scores(self):
        raw = self.raw_scores()
        q = torch.zeros_like(raw)
        for video_idx in level1_video_indices:
            if video_idx.numel() <= 1:
                continue
            x = raw.index_select(1, video_idx)
            mean = x.mean(dim=1, keepdim=True)
            centered = x - mean
            var = centered.square().mean(dim=1, keepdim=True)
            sd = torch.sqrt(var.clamp_min(SD_EPS ** 2))
            q[:, video_idx] = centered / sd
        return raw, q

model = UtilityFixedEffectModel(N_users, len(level2_ids), len(level3_ids), init_scale=INIT_SCALE).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)

print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

UtilityFixedEffectModel()
parameters: 639720


## 6. Train the Fixed Effects

In [6]:
def weighted_mse_for_indices(q_full, row_idx):
    users = obs_user_idx.index_select(0, row_idx)
    videos = obs_video_idx.index_select(0, row_idx)
    pred = q_full[users, videos]
    target = q_target.index_select(0, row_idx)
    weight = obs_weight.index_select(0, row_idx)
    return (weight * (pred - target).square()).mean(), pred, target


def eval_metrics(q_full, row_idx):
    loss, pred, target = weighted_mse_for_indices(q_full, row_idx)
    pred_np = pred.detach().cpu().numpy()
    target_np = target.detach().cpu().numpy()
    corr = float(np.corrcoef(pred_np, target_np)[0, 1]) if len(pred_np) > 2 and np.std(pred_np) > 0 and np.std(target_np) > 0 else np.nan
    return float(loss.detach().cpu()), corr

loss_history = []
best_val_loss = np.inf
best_state = None
best_epoch = 0
stale_epochs = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    _, q_full = model.normalized_scores()
    train_loss, _, _ = weighted_mse_for_indices(q_full, train_idx)
    reg_loss = LAMBDA_ALPHA * model.alpha.square().mean() + LAMBDA_DELTA * model.delta.square().mean()
    loss = train_loss + reg_loss
    loss.backward()
    optimizer.step()

    if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == NUM_EPOCHS:
        model.eval()
        with torch.no_grad():
            _, q_eval = model.normalized_scores()
            train_eval_loss, train_corr = eval_metrics(q_eval, train_idx)
            val_loss, val_corr = eval_metrics(q_eval, val_idx)
        record = {
            "epoch": epoch,
            "train_weighted_mse": train_eval_loss,
            "val_weighted_mse": val_loss,
            "train_corr_q_target": train_corr,
            "val_corr_q_target": val_corr,
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
        }
        loss_history.append(record)
        print(record)

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += LOG_EVERY
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                print(f"early stopping at epoch {epoch}; best epoch {best_epoch}, best val MSE {best_val_loss:.6g}")
                break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

loss_history_df = pd.DataFrame(loss_history)
loss_history_df.to_csv(LOSS_HISTORY_OUTPUT_PATH, index=False)
print("best epoch:", best_epoch)
print("best val weighted MSE:", best_val_loss)
print("saved loss history:", LOSS_HISTORY_OUTPUT_PATH)

{'epoch': 1, 'train_weighted_mse': 1.0574404001235962, 'val_weighted_mse': 1.1638890504837036, 'train_corr_q_target': 0.03898740222399873, 'val_corr_q_target': 0.010336411598052927, 'learning_rate': 0.01}


{'epoch': 10, 'train_weighted_mse': 0.7238360643386841, 'val_weighted_mse': 1.0154632329940796, 'train_corr_q_target': 0.08792488979537154, 'val_corr_q_target': 0.018895223244804905, 'learning_rate': 0.01}


{'epoch': 20, 'train_weighted_mse': 0.6216059327125549, 'val_weighted_mse': 1.0080832242965698, 'train_corr_q_target': 0.09350536837476041, 'val_corr_q_target': 0.01829299212994971, 'learning_rate': 0.01}


{'epoch': 30, 'train_weighted_mse': 0.5476534366607666, 'val_weighted_mse': 1.0143228769302368, 'train_corr_q_target': 0.09919888667401919, 'val_corr_q_target': 0.01780471440153724, 'learning_rate': 0.01}


{'epoch': 40, 'train_weighted_mse': 0.49916312098503113, 'val_weighted_mse': 1.0207233428955078, 'train_corr_q_target': 0.10362967092849436, 'val_corr_q_target': 0.016615374011124083, 'learning_rate': 0.01}


{'epoch': 50, 'train_weighted_mse': 0.47094109654426575, 'val_weighted_mse': 1.0240793228149414, 'train_corr_q_target': 0.10650946100901018, 'val_corr_q_target': 0.015740440604985723, 'learning_rate': 0.01}


{'epoch': 60, 'train_weighted_mse': 0.4548874795436859, 'val_weighted_mse': 1.0312079191207886, 'train_corr_q_target': 0.10874158899165474, 'val_corr_q_target': 0.015155273645298202, 'learning_rate': 0.01}
early stopping at epoch 60; best epoch 20, best val MSE 1.00808
best epoch: 20
best val weighted MSE: 1.0080832242965698
saved loss history: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_utility_loss_history.csv


## 7. Diagnostics, Random Noise, and Moment Checks

After training, compute the systematic fitted component over the full user-video catalog:

$$
q^{\mathrm{signal}}.
$$

Then draw one fixed random shock per user-video pair, standardize it within each user × level-1 category, combine it with the signal using variance share:

$$
\eta,
$$

and finally re-standardize the combined score to obtain:

$$
q^{\mathrm{true}}.
$$

The observed-row fit residual remains:

$$
q^{\mathrm{signal}}_{ij}-q^{\mathrm{target}}_{itj},
$$

because the random shock is part of the semi-synthetic true utility environment, not part of the fitted posterior-EU signal.

In [7]:
def standardize_by_level1_np(x_full, catalog_level1_idx_np, k_cat, sd_eps=1e-6):
    x_full = np.asarray(x_full, dtype=np.float32)
    out = np.empty_like(x_full, dtype=np.float32)
    for k in range(k_cat):
        video_mask = catalog_level1_idx_np == k
        n_videos = int(video_mask.sum())
        if n_videos == 0:
            continue
        if n_videos == 1:
            out[:, video_mask] = 0.0
            continue
        x_group = x_full[:, video_mask]
        mean = x_group.mean(axis=1, keepdims=True)
        centered = x_group - mean
        sd = np.sqrt((centered ** 2).mean(axis=1, keepdims=True))
        sd = np.maximum(sd, float(sd_eps))
        out[:, video_mask] = centered / sd
    return out


model.eval()
with torch.no_grad():
    raw_full_t, q_signal_t = model.normalized_scores()
    raw_full = raw_full_t.detach().cpu().numpy().astype(np.float32)
    q_signal_full = q_signal_t.detach().cpu().numpy().astype(np.float32)

rng_noise = np.random.default_rng(RANDOM_UTILITY_SEED)
eps_raw_full = rng_noise.standard_normal(size=q_signal_full.shape).astype(np.float32)
eps_std_full = standardize_by_level1_np(eps_raw_full, catalog_level1_idx_np, K_cat, SD_EPS)
eta = float(RANDOM_UTILITY_VARIANCE_SHARE)
q_raw_full = (np.sqrt(1.0 - eta) * q_signal_full + np.sqrt(eta) * eps_std_full).astype(np.float32)
q_true_full = standardize_by_level1_np(q_raw_full, catalog_level1_idx_np, K_cat, SD_EPS)

obs_q_signal = q_signal_full[obs_user_idx_np, obs_video_idx_np]
obs_eps_std = eps_std_full[obs_user_idx_np, obs_video_idx_np]
obs_q_true = q_true_full[obs_user_idx_np, obs_video_idx_np]
obs_raw_score = raw_full[obs_user_idx_np, obs_video_idx_np]
observed_df["raw_score"] = obs_raw_score.astype(np.float32)
observed_df["q_signal"] = obs_q_signal.astype(np.float32)
observed_df["eps_std"] = obs_eps_std.astype(np.float32)
observed_df["q_true"] = obs_q_true.astype(np.float32)
observed_df["q_omega"] = observed_df["q_true"]  # Backward-compatible alias for older diagnostics.
observed_df["u_star"] = (observed_df["mu_hat"].to_numpy(dtype=np.float32) + observed_df["sigma_hat"].to_numpy(dtype=np.float32) * obs_q_true).astype(np.float32)
observed_df["fit_residual"] = (observed_df["q_signal"] - observed_df["q_target"]).astype(np.float32)

moment_records = []
for k in range(K_cat):
    video_mask = catalog_level1_idx_np == k
    q_signal_group = q_signal_full[:, video_mask]
    eps_group = eps_std_full[:, video_mask]
    q_true_group = q_true_full[:, video_mask]
    raw_group = raw_full[:, video_mask]
    moment_records.append({
        "cat_idx_int": k,
        "i_cat_level1_id": int(cat_ids[k]),
        "n_videos": int(video_mask.sum()),
        "max_abs_signal_mean": float(np.max(np.abs(q_signal_group.mean(axis=1)))),
        "max_abs_signal_sd_error": float(np.max(np.abs(q_signal_group.std(axis=1, ddof=0) - 1.0))),
        "max_abs_eps_mean": float(np.max(np.abs(eps_group.mean(axis=1)))),
        "max_abs_eps_sd_error": float(np.max(np.abs(eps_group.std(axis=1, ddof=0) - 1.0))),
        "max_abs_true_mean": float(np.max(np.abs(q_true_group.mean(axis=1)))),
        "max_abs_true_sd_error": float(np.max(np.abs(q_true_group.std(axis=1, ddof=0) - 1.0))),
        "near_zero_raw_sd_users": int((raw_group.std(axis=1, ddof=0) < 1e-5).sum()),
    })
moment_df = pd.DataFrame(moment_records)

train_pred = obs_q_signal[train_idx_np]
train_target = q_target_np[train_idx_np]
val_pred = obs_q_signal[val_idx_np]
val_target = q_target_np[val_idx_np]

fit_diag = {
    "train_mse_unweighted": float(np.mean((train_pred - train_target) ** 2)),
    "val_mse_unweighted": float(np.mean((val_pred - val_target) ** 2)),
    "train_corr_q_target": float(np.corrcoef(train_pred, train_target)[0, 1]),
    "val_corr_q_target": float(np.corrcoef(val_pred, val_target)[0, 1]),
    "observed_corr_q_target": float(np.corrcoef(obs_q_signal, q_target_np)[0, 1]),
    "observed_corr_q_true_q_target": float(np.corrcoef(obs_q_true, q_target_np)[0, 1]),
    "observed_corr_q_signal_q_true": float(np.corrcoef(obs_q_signal, obs_q_true)[0, 1]),
    "observed_corr_eps_q_signal": float(np.corrcoef(obs_eps_std, obs_q_signal)[0, 1]),
    "random_utility_variance_share": eta,
    "max_abs_q_signal_mean_by_user_level1": float(moment_df["max_abs_signal_mean"].max()),
    "max_abs_q_signal_sd_error_by_user_level1": float(moment_df["max_abs_signal_sd_error"].max()),
    "max_abs_q_true_mean_by_user_level1": float(moment_df["max_abs_true_mean"].max()),
    "max_abs_q_true_sd_error_by_user_level1": float(moment_df["max_abs_true_sd_error"].max()),
    "near_zero_raw_sd_user_level1_groups": int(moment_df["near_zero_raw_sd_users"].sum()),
}
print(json.dumps(fit_diag, indent=2))
display(moment_df.sort_values("max_abs_true_sd_error", ascending=False).head(10))

bin_df = observed_df[["q_signal", "q_true", "q_target"]].copy()
bin_df["pred_decile"] = pd.qcut(bin_df["q_signal"], 10, duplicates="drop")
binned_fit = (
    bin_df
    .groupby("pred_decile", observed=True)
    .agg(
        n=("q_target", "size"),
        q_signal_mean=("q_signal", "mean"),
        q_true_mean=("q_true", "mean"),
        q_target_mean=("q_target", "mean"),
        q_target_std=("q_target", "std"),
    )
    .reset_index()
)
binned_fit.to_csv(BINNED_FIT_OUTPUT_PATH, index=False)
display(binned_fit)

{
  "train_mse_unweighted": 0.6216061115264893,
  "val_mse_unweighted": 1.008083462715149,
  "train_corr_q_target": 0.09350536837476041,
  "val_corr_q_target": 0.01829299212994971,
  "observed_corr_q_target": 0.08841832741778381,
  "observed_corr_q_true_q_target": 0.08193256366484418,
  "observed_corr_q_signal_q_true": 0.9251369312762817,
  "observed_corr_eps_q_signal": -0.00028209778121406504,
  "random_utility_variance_share": 0.05,
  "max_abs_q_signal_mean_by_user_level1": 0.014901161193847656,
  "max_abs_q_signal_sd_error_by_user_level1": 1.0,
  "max_abs_q_true_mean_by_user_level1": 1.324286387216489e-07,
  "max_abs_q_true_sd_error_by_user_level1": 1.3113021850585938e-06,
  "near_zero_raw_sd_user_level1_groups": 1777
}


,cat_idx_int,i_cat_level1_id,n_videos,max_abs_signal_mean,max_abs_signal_sd_error,max_abs_eps_mean,max_abs_eps_sd_error,max_abs_true_mean,max_abs_true_sd_error,near_zero_raw_sd_users
28,28,28,1095,1.071332e-06,0.000006,1.485489e-07,1.072884e-06,9.161153e-08,1.311302e-06,0
33,33,34,913,1.584944e-04,0.000003,1.051731e-07,1.072884e-06,5.039955e-08,1.132488e-06,0
8,8,8,824,1.896308e-06,0.000007,1.171839e-07,1.013279e-06,1.324286e-07,9.536743e-07,0
5,5,5,1034,3.676508e-07,0.000005,1.090998e-07,1.072884e-06,1.070463e-07,8.940697e-07,0
19,19,19,532,2.655740e-07,0.000003,1.192653e-07,8.940697e-07,6.033289e-08,8.344650e-07,0
7,7,7,317,2.713704e-07,0.000002,1.154017e-07,6.556511e-07,6.026274e-08,8.344650e-07,0
25,25,25,363,4.008125e-07,0.000003,1.190451e-07,6.556511e-07,7.790526e-08,8.344650e-07,0
32,32,33,379,1.490116e-02,1.000000,1.239273e-07,8.344650e-07,9.412010e-08,7.748604e-07,1777
6,6,6,471,5.072089e-07,0.000003,1.250306e-07,7.152557e-07,6.099668e-08,7.152557e-07,0
9,9,9,366,5.603814e-07,0.000003,9.986780e-08,7.152557e-07,7.401724e-08,7.152557e-07,0


,pred_decile,n,q_signal_mean,q_true_mean,q_target_mean,q_target_std
0,"(-15.383, -0.592]",459987,-1.150682,-1.121719,0.014862,0.502704
1,"(-0.592, -0.351]",405223,-0.483105,-0.471059,0.022243,0.490907
2,"(-0.351, -0.153]",432479,-0.241199,-0.235583,0.022876,0.495930
3,"(-0.153, -0.0249]",432538,-0.086376,-0.083726,0.053837,0.480491
4,"(-0.0249, 0.0529]",432555,0.010268,0.007207,0.095121,0.465383
5,"(0.0529, 0.16]",432597,0.106038,0.102909,0.110244,0.447993
6,"(0.16, 0.279]",432517,0.216920,0.210835,0.144875,0.429528
7,"(0.279, 0.438]",432557,0.355030,0.346237,0.168385,0.422325
8,"(0.438, 0.592]",433919,0.534570,0.520328,0.141039,0.439702
9,"(0.592, 14.315]",431188,1.201190,1.170756,0.163991,0.435685


## 8. Save Parameters and Observed-Row Utility Output

The saved observed-row table distinguishes the fitted systematic score:

$$
q^{\mathrm{signal}},
$$

the standardized random shock:

$$
\epsilon^{\mathrm{std}},
$$

and the final noisy true utility score:

$$
q^{\mathrm{true}}.
$$

The residual is always computed against the fitted signal:

$$
\text{fit residual}=q^{\mathrm{signal}}_{ij}-q^{\mathrm{target}}_{itj}.
$$

In [8]:
alpha_np = model.alpha.detach().cpu().numpy().astype(np.float32)
delta_np = model.delta.detach().cpu().numpy().astype(np.float32)

np.savez_compressed(
    PARAM_OUTPUT_NPZ,
    alpha=alpha_np,
    delta=delta_np,
    user_ids=user_ids.astype(np.int64),
    level2_ids=level2_ids.astype(np.int64),
    level3_ids=level3_ids.astype(np.int64),
    cat_ids=cat_ids.astype(np.int64),
    catalog_video_ids=catalog_df["video_id"].to_numpy(dtype=np.int64),
    catalog_level1_ids=catalog_df["i_cat_level1_id"].to_numpy(dtype=np.int64),
    catalog_level2_ids=catalog_df["i_cat_level2_id"].to_numpy(dtype=np.int64),
    catalog_level3_ids=catalog_df["i_cat_level3_id"].to_numpy(dtype=np.int64),
    catalog_cat_idx=catalog_level1_idx_np.astype(np.int64),
    catalog_level2_idx=catalog_level2_idx_np.astype(np.int64),
    catalog_level3_idx=catalog_level3_idx_np.astype(np.int64),
    mu_matrix=mu_matrix.astype(np.float32),
    sigmas_by_user=sigmas_by_user.astype(np.float32),
    random_utility_variance_share=np.array([RANDOM_UTILITY_VARIANCE_SHARE], dtype=np.float32),
    random_utility_seed=np.array([RANDOM_UTILITY_SEED], dtype=np.int64),
    seed=np.array([SEED], dtype=np.int64),
)

torch.save({
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "config": {
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "random_utility_variance_share": RANDOM_UTILITY_VARIANCE_SHARE,
        "random_utility_seed": RANDOM_UTILITY_SEED,
        "lambda_alpha": LAMBDA_ALPHA,
        "lambda_delta": LAMBDA_DELTA,
        "init_scale": INIT_SCALE,
        "sd_eps": SD_EPS,
        "validation_share": VALIDATION_SHARE,
        "use_confidence_weights": USE_CONFIDENCE_WEIGHTS,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
    },
    "user_ids": torch.from_numpy(user_ids.astype(np.int64)),
    "level2_ids": torch.from_numpy(level2_ids.astype(np.int64)),
    "level3_ids": torch.from_numpy(level3_ids.astype(np.int64)),
    "cat_ids": torch.from_numpy(cat_ids.astype(np.int64)),
    "catalog_video_ids": torch.from_numpy(catalog_df["video_id"].to_numpy(dtype=np.int64)),
    "catalog_cat_idx": torch.from_numpy(catalog_level1_idx_np.astype(np.int64)),
    "catalog_level2_idx": torch.from_numpy(catalog_level2_idx_np.astype(np.int64)),
    "catalog_level3_idx": torch.from_numpy(catalog_level3_idx_np.astype(np.int64)),
}, CHECKPOINT_OUTPUT_PATH)

observed_output_cols = [
    "row_id", "user_id", "video_id", "i_cat_level1_id", "i_cat_level2_id", "i_cat_level3_id",
    "r_hat", "tau_hat", "mu_hat", "sigma_hat", "EU", "q_target",
    "raw_score", "q_signal", "eps_std", "q_true", "q_omega", "u_star", "fit_residual", "posterior_confidence_weight",
]
observed_df[observed_output_cols].to_parquet(OBSERVED_UTILITY_OUTPUT_PATH, index=False)

print("saved params:", PARAM_OUTPUT_NPZ)
print("saved checkpoint:", CHECKPOINT_OUTPUT_PATH)
print("saved observed utility output:", OBSERVED_UTILITY_OUTPUT_PATH)

saved params: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_utility_params.npz
saved checkpoint: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_utility_fe_model.pt
saved observed utility output: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_utility_observed.parquet


## 9. Save Full User-Video Utility Catalog

This table is large:

$$
1777\times 10728
$$

rows. It is the artifact later ranking simulations should use as the known true utility surface.

The catalog output includes the systematic fitted signal, the fixed standardized random shock, the final noisy standardized true utility score, and the calibrated true utility:

$$
u^*_{ij}=\hat\mu_{i,k_1(j)}+\hat\sigma_{i,k_1(j)}q^{\mathrm{true}}_{ij}.
$$

In [9]:
def make_catalog_chunk(user_start, user_end):
    user_slice = slice(user_start, user_end)
    n_chunk_users = user_end - user_start
    n_videos = len(catalog_df)
    mu_chunk = mu_matrix[user_slice][:, catalog_level1_idx_np]
    sigma_chunk = sigmas_by_user[user_slice][:, catalog_level1_idx_np]
    u_star_chunk = mu_chunk + sigma_chunk * q_true_full[user_slice]
    return pd.DataFrame({
        "user_id": np.repeat(user_ids[user_slice], n_videos).astype(np.int64),
        "video_id": np.tile(catalog_df["video_id"].to_numpy(dtype=np.int64), n_chunk_users),
        "level1_category": np.tile(catalog_df["i_cat_level1_id"].to_numpy(dtype=np.int64), n_chunk_users),
        "level2_category": np.tile(catalog_df["i_cat_level2_id"].to_numpy(dtype=np.int64), n_chunk_users),
        "level3_category": np.tile(catalog_df["i_cat_level3_id"].to_numpy(dtype=np.int64), n_chunk_users),
        "raw_score": raw_full[user_slice].reshape(-1).astype(np.float32),
        "q_signal": q_signal_full[user_slice].reshape(-1).astype(np.float32),
        "eps_std": eps_std_full[user_slice].reshape(-1).astype(np.float32),
        "q_true": q_true_full[user_slice].reshape(-1).astype(np.float32),
        "q_omega": q_true_full[user_slice].reshape(-1).astype(np.float32),
        "mu_hat": mu_chunk.reshape(-1).astype(np.float32),
        "sigma_hat": sigma_chunk.reshape(-1).astype(np.float32),
        "u_star": u_star_chunk.reshape(-1).astype(np.float32),
    })

if WRITE_FULL_CATALOG_OUTPUT:
    import pyarrow as pa
    import pyarrow.parquet as pq

    if TRUE_UTILITY_OUTPUT_PATH.exists():
        if TRUE_UTILITY_OUTPUT_PATH.is_dir():
            shutil.rmtree(TRUE_UTILITY_OUTPUT_PATH)
        else:
            TRUE_UTILITY_OUTPUT_PATH.unlink()

    if CATALOG_OUTPUT_MODE == "single_file":
        writer = None
        try:
            for user_start in range(0, N_users, CATALOG_USER_CHUNK_SIZE):
                user_end = min(N_users, user_start + CATALOG_USER_CHUNK_SIZE)
                chunk_df = make_catalog_chunk(user_start, user_end)
                table = pa.Table.from_pandas(chunk_df, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(TRUE_UTILITY_OUTPUT_PATH, table.schema, compression="zstd")
                writer.write_table(table)
                print(f"wrote users [{user_start}, {user_end})")
        finally:
            if writer is not None:
                writer.close()
    elif CATALOG_OUTPUT_MODE == "partitioned_by_user":
        TRUE_UTILITY_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
        for user_start in range(0, N_users, CATALOG_USER_CHUNK_SIZE):
            user_end = min(N_users, user_start + CATALOG_USER_CHUNK_SIZE)
            chunk_df = make_catalog_chunk(user_start, user_end)
            table = pa.Table.from_pandas(chunk_df, preserve_index=False)
            pq.write_to_dataset(table, root_path=str(TRUE_UTILITY_OUTPUT_PATH), partition_cols=["user_id"], compression="zstd")
            print(f"wrote users [{user_start}, {user_end})")
    else:
        raise ValueError(f"Unknown CATALOG_OUTPUT_MODE: {CATALOG_OUTPUT_MODE}")

    print("saved full utility catalog:", TRUE_UTILITY_OUTPUT_PATH)
else:
    print("WRITE_FULL_CATALOG_OUTPUT is False; skipped full utility catalog write")

wrote users [0, 64)


wrote users [64, 128)


wrote users [128, 192)


wrote users [192, 256)


wrote users [256, 320)


...

wrote users [1472, 1536)


wrote users [1536, 1600)


wrote users [1600, 1664)


wrote users [1664, 1728)


wrote users [1728, 1777)
saved full utility catalog: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_true_utility.parquet


## 10. Save Diagnostics

In [10]:
diag = {
    "target_path": str(OBSERVED_TARGET_PATH),
    "processed_data_path": str(PROCESSED_DATA_PATH),
    "n_users": int(N_users),
    "n_level1_categories": int(K_cat),
    "n_catalog_videos": int(len(catalog_df)),
    "n_level2_categories": int(len(level2_ids)),
    "n_level3_categories": int(len(level3_ids)),
    "n_observed_rows": int(len(observed_df)),
    "n_dropped_observed_rows": int(n_dropped),
    "category_conflict_counts": conflict_counts,
    "config": {
        "num_epochs": int(NUM_EPOCHS),
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "random_utility_variance_share": float(RANDOM_UTILITY_VARIANCE_SHARE),
        "random_utility_seed": int(RANDOM_UTILITY_SEED),
        "lambda_alpha": float(LAMBDA_ALPHA),
        "lambda_delta": float(LAMBDA_DELTA),
        "init_scale": float(INIT_SCALE),
        "sd_eps": float(SD_EPS),
        "validation_share": float(VALIDATION_SHARE),
        "use_confidence_weights": bool(USE_CONFIDENCE_WEIGHTS),
        "device": DEVICE,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
    },
    "fit": fit_diag,
    "moment_diagnostics": {
        "max_abs_q_signal_mean_by_user_level1": float(moment_df["max_abs_signal_mean"].max()),
        "max_abs_q_signal_sd_error_by_user_level1": float(moment_df["max_abs_signal_sd_error"].max()),
        "max_abs_q_true_mean_by_user_level1": float(moment_df["max_abs_true_mean"].max()),
        "max_abs_q_true_sd_error_by_user_level1": float(moment_df["max_abs_true_sd_error"].max()),
        "near_zero_raw_sd_user_level1_groups": int(moment_df["near_zero_raw_sd_users"].sum()),
    },
    "outputs": {
        "params_npz": str(PARAM_OUTPUT_NPZ),
        "checkpoint": str(CHECKPOINT_OUTPUT_PATH),
        "observed_utility": str(OBSERVED_UTILITY_OUTPUT_PATH),
        "true_utility_catalog": str(TRUE_UTILITY_OUTPUT_PATH) if WRITE_FULL_CATALOG_OUTPUT else None,
        "loss_history": str(LOSS_HISTORY_OUTPUT_PATH),
        "binned_fit": str(BINNED_FIT_OUTPUT_PATH),
    },
}
DIAGNOSTIC_OUTPUT_PATH.write_text(json.dumps(diag, indent=2))
print("saved diagnostics:", DIAGNOSTIC_OUTPUT_PATH)
print(json.dumps(diag, indent=2)[:4000])

saved diagnostics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_utility_diagnostics.json
{
  "target_path": "/Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_observed_targets.parquet",
  "processed_data_path": "/Users/haozhangao/Desktop/RecSys Research/KuaiRec 2.0/data/processed/processed_data.parquet",
  "n_users": 1777,
  "n_level1_categories": 39,
  "n_catalog_videos": 10728,
  "n_level2_categories": 139,
  "n_level3_categories": 221,
  "n_observed_rows": 4325560,
  "n_dropped_observed_rows": 0,
  "category_conflict_counts": {
    "i_cat_level1_id": 0,
    "i_cat_level2_id": 0,
    "i_cat_level3_id": 0
  },
  "config": {
    "num_epochs": 300,
    "learning_rate": 0.01,
    "weight_decay": 0.0,
    "random_utility_variance_share": 0.05,
    "random_utility_seed": 20261430,
    "lambda_alpha": 0.0,
    "lambda_delta": 0.0,
    "init_scale": 0.01,
    "sd_eps": 1e-06,
    "validation_share": 0.05,
    "use_confidence_weights"

## 12. Utility Correlation Diagnostics

This final check compares the calibrated true utility against posterior expected utility and observed behavior metrics. Re-run this cell after changing the random utility variance share to see how the signal/noise calibration changes the correlations.

In [11]:
UTILITY_CORR_OUTPUT_PATH = OUTPUT_DIR / "semi_synth_utility_metric_correlations.csv"

utility_corr_cols = [
    "row_id", "user_id", "video_id", "EU", "q_target", "r_hat",
    "raw_score", "q_signal", "eps_std", "q_true", "u_star",
]

if "observed_df" in globals() and set(utility_corr_cols).issubset(observed_df.columns):
    utility_corr_df = observed_df[utility_corr_cols].copy()
    utility_corr_source = "in-memory observed_df"
else:
    utility_corr_df = pd.read_parquet(OBSERVED_UTILITY_OUTPUT_PATH, columns=utility_corr_cols)
    utility_corr_source = str(OBSERVED_UTILITY_OUTPUT_PATH)

n_u_star = int(utility_corr_df["u_star"].replace([np.inf, -np.inf], np.nan).notna().sum())
if n_u_star == 0:
    raise ValueError(
        "No non-missing u_star values are available. Re-run notebook 08 from the top after the CPU DEVICE fix; "
        "the current observed utility artifact was produced by an unstable/failed run."
    )
target_log_wr_df = pd.read_parquet(OBSERVED_TARGET_PATH, columns=["row_id", "log_watch_ratio"])

row_ids = utility_corr_df["row_id"].to_numpy(dtype=np.int64)
behavior_cols = ["watch_ratio", "play_duration", "y_complete", "y_long", "y_rewatch", "y_neg"]
behavior_df = pd.read_parquet(PROCESSED_DATA_PATH, columns=behavior_cols).iloc[row_ids].reset_index(drop=True)
behavior_df["row_id"] = row_ids

corr_data = utility_corr_df.merge(target_log_wr_df, on="row_id", how="left", validate="one_to_one")
corr_data = corr_data.merge(behavior_df, on="row_id", how="left", validate="one_to_one")
corr_data["log1p_watch_ratio"] = np.log1p(
    pd.to_numeric(corr_data["watch_ratio"], errors="coerce").clip(lower=0)
)

metric_specs = [
    ("EU", "posterior expected utility"),
    ("q_target", "standardized posterior EU"),
    ("raw_score", "raw structural video score"),
    ("q_signal", "normalized structural video score"),
    ("watch_ratio", "watch ratio"),
    ("log_watch_ratio", "stored log watch ratio"),
    ("log1p_watch_ratio", "log(1 + watch ratio)"),
    ("play_duration", "watch duration"),
    ("y_complete", "completion"),
    ("y_long", "long-watch"),
    ("y_rewatch", "rewatch"),
    ("y_neg", "negative/short-watch"),
    ("r_hat", "posterior responsibility"),
    ("eps_std", "random utility shock"),
    ("q_true", "standardized true utility sanity check"),
]

corr_rows = []
for metric, description in metric_specs:
    sub = corr_data[["u_star", metric]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(sub) < 3 or sub["u_star"].std(ddof=0) == 0 or sub[metric].std(ddof=0) == 0:
        pearson_corr = np.nan
        spearman_corr = np.nan
    else:
        pearson_corr = float(sub["u_star"].corr(sub[metric], method="pearson"))
        spearman_corr = float(sub["u_star"].corr(sub[metric], method="spearman"))
    corr_rows.append({
        "metric": metric,
        "description": description,
        "n": int(len(sub)),
        "pearson_corr_with_u_star": pearson_corr,
        "spearman_corr_with_u_star": spearman_corr,
    })

utility_corrs = pd.DataFrame(corr_rows)
utility_corrs.to_csv(UTILITY_CORR_OUTPUT_PATH, index=False)

print("random_utility_variance_share:", RANDOM_UTILITY_VARIANCE_SHARE)
print("random_utility_seed:", RANDOM_UTILITY_SEED)
print("utility source:", utility_corr_source)
print("rows:", len(corr_data), "users:", corr_data["user_id"].nunique(), "videos:", corr_data["video_id"].nunique())
print("saved correlations:", UTILITY_CORR_OUTPUT_PATH)
display(utility_corrs)


random_utility_variance_share: 0.05
random_utility_seed: 20261430
utility source: in-memory observed_df
rows: 4325560 users: 1777 videos: 9161
saved correlations: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/semi_synth_utility_metric_correlations.csv


,metric,description,n,pearson_corr_with_u_star,spearman_corr_with_u_star
0,EU,posterior expected utility,4325560,0.899495,0.897747
1,q_target,standardized posterior EU,4325560,0.169982,0.073763
2,raw_score,raw structural video score,4325560,0.260224,0.259809
3,q_signal,normalized structural video score,4325560,0.393825,0.377759
4,watch_ratio,watch ratio,4325560,0.021230,0.160213
5,log_watch_ratio,stored log watch ratio,4325560,0.218225,0.160212
6,log1p_watch_ratio,log(1 + watch ratio),4325560,0.132415,0.160213
7,play_duration,watch duration,4325560,-0.000202,0.100316
8,y_complete,completion,4325560,0.039285,0.025384
9,y_long,long-watch,4325560,-0.097067,-0.119157
